# Xây dựng mô hình Baseline

**Mục tiêu:**
- Thực hiện các bước xây dựng mô hình cơ sở nhằm tối ưu luồng pineline, đồng thời xây dựng các hàm hỗ trợ
- Xây dựng hàm đánh giá chỉ số mô hình theo tiêu chí `ISO/SEC 30107`
- Xác định mục tiêu và hướng đi cho toàn bộ dự án
- Sử dụng mô hình cơ sở để đánh giá thêm một vài phương pháp huấn luyện mô hình khác được kiểm tra trong notebooks [03_Optimize.ipynb](./03_Optimize.ipynb)

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, roc_curve, confusion_matrix

import tensorflow as tf
from keras.models import Sequential
from keras.layers import Rescaling, RandomFlip, RandomRotation, RandomZoom, RandomBrightness, RandomContrast, RandomTranslation
from keras.optimizers import Adam
from keras.utils import image_dataset_from_directory, set_random_seed
from keras.callbacks import ModelCheckpoint, EarlyStopping

# Thêm thư mục gốc vào hệ thống để import code tự viết
sys.path.append(os.path.abspath('..'))
from models.BaselineModel import baseline_model
from src.evaluate import MetricsCallback

In [ ]:
IMG_SIZE = (224, 224)
EPOCHS = 30
BATCH_SIZE = 32
SEED = 42
OUTPUT_DIR = '../outputs/baseline-model'

set_random_seed(SEED)

## 1. Chuẩn bị dữ liệu

---

In [8]:
train_dir = '../datasets/LCC_FASD/train'
val_dir = '../datasets/LCC_FASD/val'

# 1. Load dữ liệu thô từ thư mục
print("Tập Train:")
train_ds = image_dataset_from_directory(
    train_dir, 
    image_size=IMG_SIZE, 
    batch_size=BATCH_SIZE, 
    label_mode='binary',
    shuffle=True,
    seed=SEED
)

print("\nTập Validation:")
val_ds = image_dataset_from_directory(
    val_dir, 
    image_size=IMG_SIZE, 
    batch_size=BATCH_SIZE, 
    label_mode='binary',
    shuffle=False,
)

Tập Train:
Found 8299 files belonging to 2 classes.

Tập Validation:
Found 2948 files belonging to 2 classes.


In [9]:
train_aug = Sequential([
    Rescaling(1./255),                  # Chuẩn hóa pixel
    RandomFlip("horizontal"),           # Lật ngang
    RandomRotation(0.05),               # Xoay ngẫu nhiên 5%
    RandomZoom(0.2),                    # Thu phóng 20%
    RandomBrightness(0.2),              # Thay đổi độ sáng 20%
    RandomContrast(0.2),                # Thay đổi tương phản 20%
    RandomTranslation(0.01, 0.01)       # Dịch chuyển 1%
])

val_aug = Sequential([Rescaling(1./255)]) # Chuẩn hóa pixel cho tập validation

AUTOTUNE = tf.data.AUTOTUNE # Tăng tốc độ tải dữ liệu bằng cách sử dụng prefetching

train_ds = train_ds.map(lambda x, y: (train_aug(x, training=True), y), num_parallel_calls=AUTOTUNE)
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)

val_ds = val_ds.map(lambda x, y: (val_aug(x, training=True), y), num_parallel_calls=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)


## 2. Xây dựng mô hình

---

In [10]:
input_shape = (IMG_SIZE[0], IMG_SIZE[1], 3)

model = baseline_model(input_shape=input_shape, num_classes=1)

model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,557,249 (25.01 MB)

 Trainable params: 6,557,249 (25.01 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
metrics_callback = MetricsCallback(val_dataset=val_ds)

callbacks = [
    metrics_callback,

    ModelCheckpoint(
        filepath=OUTPUT_DIR + '/best_model.keras', 
        monitor='val_acer', 
        save_best_only=True, 
        mode='min', 
        verbose=1
    ),

    EarlyStopping(
        monitor='val_acer',
        patience=5,
        mode='min',
        verbose=1,
        restore_best_weights=True
    )
]

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## 3. Huấn luyện mô hình

---

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/30


/home/dainguyen1506/Documents/Project/fas/.venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step - accuracy: 0.8428 - loss: 0.5403 - val_eer: 53.38% - val_acer: 53.38% (t*: 0.7975)

Epoch 1: val_acer improved from None to 0.53382, saving model to ../outputs/baseline-model/best_model.keras

Epoch 1: finished saving model to ../outputs/baseline-model/best_model.keras
260/260 ━━━━━━━━━━━━━━━━━━━━ 56s 208ms/step - accuracy: 0.8428 - loss: 0.5403 - val_accuracy: 0.8626 - val_loss: 0.4190 - val_eer: 0.5338 - val_acer: 0.5338
Epoch 2/30
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step - accuracy: 0.8526 - loss: 0.4279 - val_eer: 48.32% - val_acer: 48.32% (t*: 0.8512)

Epoch 2: val_acer improved from 0.53382 to 0.48323, saving model to ../outputs/baseline-model/best_model.keras

Epoch 2: finished saving model to ../outputs/baseline-model/best_model.keras
260/260 ━━━━━━━━━━━━━━━━━━━━ 47s 181ms/step - accuracy: 0.8526 - loss: 0.4279 - val_accuracy: 0.8626 - val_loss: 0.4005 - val_eer: 0.4832 - val_acer: 0.4832
Epoch 3/30
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 1